# FINAL PROJECT - UNITED WAY OF SOUTHERN MAINE

#### author: Muhammad Marzouk Baig
#### last edited: 04/01/2026

# Clean & Load

*   Load
*   Clean
*   Rename Columns




In [85]:
import pandas as pd

In [86]:
df = pd.read_excel('UWSM Community Data 4-1-2026.xlsx', sheet_name='Data')

In [87]:
# standardizing column names
df.columns = [
    'uid',
    'survey_type',
    'start_time',
    'biggest_challenge',
    'live_work_sm',
    'alice',
    'feel_heard',
    'community',
    'age_group',
    'race_ethnicity',
    'relation_uwsm',
    'want_involvement',
    'how_involvement',
    'hardest_expenses',
    'zip_code',
    'employed',
    'employment_type',
    'roadblock_expenses',
    'helpful_supports'
]
print(df.head())

     uid survey_type          start_time biggest_challenge  \
0  CASH1        CASH 45889 days 16:18:22               NaN   
1  CASH2        CASH 45889 days 16:18:49               NaN   
2  CASH3        CASH 45889 days 16:43:22               NaN   
3  CASH4        CASH 45889 days 16:57:55               NaN   
4  CASH5        CASH 45889 days 18:58:47               NaN   

         live_work_sm                   alice feel_heard  \
0  Other Maine County                     Yes        Yes   
1  Other Maine County  Maybe, with difficulty      Maybe   
2  Other Maine County                      No        Yes   
3  Other Maine County  Maybe, with difficulty         No   
4  Other Maine County                      No        Yes   

                                           community age_group race_ethnicity  \
0  65 plus We often have reduced income but with ...       NaN            NaN   
1                                the Village of Gray       NaN            NaN   
2                      

# Classifying open ended responses

In [88]:
from transformers import pipeline
from multiprocessing import Pool
import spacy

In [89]:
# Initialize classifier
classifier = pipeline(
    "zero-shot-classification",
    model="facebook/bart-large-mnli"
)

Loading weights:   0%|          | 0/515 [00:00<?, ?it/s]

In [90]:
# process a single row this function called
# semicolon seperated string returned.
def process_row(text, mapping, categories):
    if not text or text.strip() == "":
        return ""  # keep empty / missing separate

    parts = text.split(';')
    structured = []
    unstructured = []

    for p in parts:
        p_clean = p.strip()
        if mapping and p_clean in mapping:
            structured.append(mapping[p_clean])
        else:
            unstructured.append(p_clean)

    new_labels = []
    for t in unstructured:
        if t:
            result = classifier(t, candidate_labels=categories, multi_label=True)
            # lower threshold to catch weak matches
            labels = [l for l, s in zip(result['labels'], result['scores']) if s >= 0.1][:2]

            # fallback: anything unmapped or weak goes into 'Other'
            if not labels:
                labels = ["Other"]
            new_labels.extend(labels)

    final = list(set(structured + new_labels))
    return ';'.join(final)

In [91]:
def process_multi(text, mapping):
    if not text:
        return ""
    parts = text.split(';')
    mapped = [mapping.get(p.strip(), None) for p in parts]
    mapped = [m for m in mapped if m]
    return ';'.join(set(mapped))

In [92]:
mapping_biggest_challenge = {
    "Paying for housing (rent, utilities, mortgage, taxes)": "Housing Cost",
    "Covering basic expenses (groceries, gas, emergencies)": "Cost of Basics",
    "Getting enough food": "Food Access",
    "Finding affordable healthcare": "Healthcare Cost",
    "Mental health challenges (like anxiety, depression)": "Mental Health",
    'Behavioral health challenges (like substance use or addiction)': "Behavioral Health",
    'Child care that is affordable and available': "Affordable Child Care Access",
    'Caring for aging parents or relatives':'Affordable Elder Care Access',
    "Getting to work, school, or appointments (transportation)": 'Transportation',
    'Jobs with low pay or no benefits':'Low Wages',
    'Lack of stable jobs with career growth':'Job Instability',
    'Feeling isolated or not connected to community': 'Social Isolation',
    'Access to college or training after high school': 'Education Access',
    'Impact of flooding and other extreme weather':'Climate Impacts',
}
categories_biggest_challenge = [
'Affordable Housing Availability',
'Language Access',
'Domestic Violence',
'Rising Taxes',
'Political Division',
'Immigration Enforcement',
'Welfare Dependence',
'Lack of in Home Support',
'Gas Costs',
'Rent Increases',
'None',
'Other',
]

mapping_live_work_sm = {

    "Cumberland": "Cumberland",
    "Cumberland & York": "Cumberland & York",
    "York": "York",
    "Other Maine County": "Outside Southern Maine"
    # "Androscoggin": "Outside Southern Maine",
    # "Franklin County": "Outside Southern Maine",
    # "Kennebec": "Outside Southern Maine",
    # "Knox": "Outside Southern Maine",
    # "Lincoln": "Outside Southern Maine",
    # "Other": "Outside Southern Maine",
    # "Oxford": "Outside Southern Maine",
    # "Penobscot": "Outside Southern Maine",
    # "Sagadahoc": "Outside Southern Maine",
    # "Somerset": "Outside Southern Maine"
}

mapping_feel_heard = {
    "Maybe": "Unheard",
    "No": "Unheard",
    "Yes": "Heard"
}

mapping_community = {
    "African American": "African American",
    "Asian": "Asian",
    "Everyone": "Other",
    "Foster kids": "Foster Kids",
    "Hispanic": "Hispanic",
    "Immigrant": "Immigrant",
    "Indigenous": "Indigenous",
    "LGBTQ+": "LGBTQ+",
    "Low Income": "ALICE",
    "Noble School": "Youth (under 18)",
    "Older adults": "Older Adults (+65)",
    "Parents": "Parents",
    "Recovery community": "Recovery Community",
    "Single mothers": "Single Mothers",
    "Town": "Town of Residence",
    "White": "White",
    "Women": "Women",
    "Youth": "Youth (under 18)"
}

mapping_alice ={
    "Maybe, with difficulty": "Below ALICE",
    "No": "Below ALICE",
    "Yes": "Above ALICE"
}


mapping_age_group = {
    "Under 18": "Under 24",
    "18 - 24": "Under 24",
    "25 - 44": "25-64",
    "45 - 64": "25-64",
    "65 - 75": "Over 65+",
    "More than 75": "Over 65+",
    "Prefer not to answer": None  # using None instead of 'null' in Python
}

mapping_race_ethnicity = {
    "American Indian or Alaska Native": "BIPOC",
    "Asian": "BIPOC",
    "Black or African American": "BIPOC",
    "Hispanic": "BIPOC",
    "More than one": "More than one",
    "Native Hawaiian or Pacific Islander": "More than one",
    "Other": "Other",
    "Prefer not to answer": "Prefer not to answer",
    "White": "White"
}

mapping_relation_uwsm = {
    "Formerly Involved": "Formerly Connected",
    "I am closely connected (I am a UWSM volunteer, staff, or partner organization)": "Closely Connected",
    "I am moderately connected (For example, I occasionally volunteer and/or donate to UWSM)": "Moderately Connected",
    "I am not connected with United Way": "Not Connected"
}

mapping_want_involvement = {
    "Yes": "Yes",
    "No": "No"
}

mapping_how_involvement = {
    "Donating money to United Way of Southern Maine": "Donate",
    "Helping connect people to services or resources": "Connect",
    "Speaking up or helping spread the word about important issues": "Speak Up",
    "Joining a group that solves community issues": "Join Group",
    "Sharing my story or experiences to help others understand": "Share Story",
    "Volunteering my time or skills in local programs": "Volunteer"
}

mapping_hardest_expenses = {
    "Credit Card or Loan Payments": "Credit Card or Loan Payments",
    "Groceries/Food": "Groceries/Food",
    "Medical bills/Healthcare Costs": "Medical bills/Healthcare Costs",
    "Housing costs (e.g., rent, mortgage)": "Housing costs (e.g., rent, mortgage)",
    "Childcare or School-Related Expenses": "Childcare or School-Related Expenses",
    "Insurance Premiums (Health, Car, Home, etc)": "Insurance Premiums (Health, Car, Home, etc)",
    "Phone or Internet Service": "Phone or Internet Service",
    "Utilities (e.g., electricity, water, gas)": "Utilities (e.g., electricity, water, gas)",
    "Transportation (e.g., car payments, gas, car repairs or public transport)": "Transportation (e.g., car payments, gas, car repairs or public transport)"
}

mapping_employed = {
    "Yes": "Yes",
    "No": "No"
}

mapping_employment_type = {
    "32 hours": "Full-Time - 1 Job",
    "a mixture of part time and self-employed": "Other",
    "Freelance/contract": "Other",
    "Full-time hours (40 or more) working for more than one employer": "Full-Time - Multiple Jobs",
    "Full-time hours (40 or more) working for one employer": "Full-Time - 1 Job",
    "I am election clerk working 3 times a year": "Other",
    "Part-time hours (29 or less)": "Part-Time",
    "Per diem": "Per Diem",
    "Self employed": "Self Employed"
}

mapping_roadblock_expenses = {
    "I don’t have reliable transportation": "Lack of Reliable Transportation",
    "I’m caring for a family member": "Family Care Responsibilities",
    "I have health issues or a disability": "Health Issues or Disability",
    "I don’t have time to go back to school or get more training": "Lack of Time for Training/Education",
    "I don’t have the skills or training needed for a better-paying job": "Lack of Skills or Training",
    "I’ve been turned away from jobs because of my background": "Turned Away",
    "I don’t have affordable child care": "Lack of Affordable Child Care",
    "I’m not sure what opportunities are available": "Unaware of or Limited Job Opportunities",
}
categories_roadblock_expenses = [
    "Age-Related Employment Barriers",
    "Insufficient Income / Fixed Income",
    "High Cost of Living",
    "Retirement Challenges",
    "Debt / Cash Flow Issues",
    "Financially Stable / Doing Well"
]

mapping_helpful_supports = {
    "Help with food or groceries": "Food",
    "Help affording healthcare or mental health support": "Health/Mental Health",
    "Help with transportation (gas, car repair, bus passes, etc.)": "Transportation",
    "Help with job training or education": "Job Training/Education",
    "Help managing debt or improving credit": "Debt/Credit",
    "Employment navigation": "Employment",
    "Rental assistance": "Rental Assistance",
    "Help navigating programs like SNAP, TANF": "SNAP/TANF Navigation",
}
categories_helpful_supports = [
  "Increased Income",
  "Taxes",
  "Home Ownership"
]


In [93]:
# config for all columns, to allow organized processing
column_configs = {
    "biggest_challenge": {
        "type": "mixed",
        "mapping": mapping_biggest_challenge,
        "categories": categories_biggest_challenge
    },
    "live_work_sm": {
        "type": "multi",
        "mapping": mapping_live_work_sm
    },
    "alice": {
        "type": "single",
        "mapping": mapping_alice
    },
    "feel_heard": {
        "type": "single",
        "mapping": mapping_feel_heard
    },
    "age_group": {
        "type": "single",
        "mapping": mapping_age_group
    },
    "race_ethnicity": {
        "type": "multi",
        "mapping": mapping_race_ethnicity
    },
    "relation_uwsm": {
        "type": "single",
        "mapping": mapping_relation_uwsm
    },
    "want_involvement": {
        "type": "single",
        "mapping": mapping_want_involvement
    },
    "how_involvement": {
        "type": "multi",
        "mapping": mapping_how_involvement
    },
    "hardest_expenses": {
        "type": "multi",
        "mapping": mapping_hardest_expenses,
    },
    "employed": {
        "type": "single",
        "mapping": mapping_employed
    },
    "employment_type": {
        "type": "single",
        "mapping": mapping_employment_type
    },
    "roadblock_expenses": {
        "type": "mixed",
        "mapping": mapping_roadblock_expenses,
        "categories": categories_roadblock_expenses
    },
    "helpful_supports": {
        "type": "mixed",
        "mapping": mapping_helpful_supports,
        "categories": categories_helpful_supports
    }
}

In [94]:
all_dummies = []

for col, config in column_configs.items():

    if config["type"] == "single":
        df[f"{col}_clean"] = df[col].map(config["mapping"])

    elif config["type"] == "multi":
        df[f"{col}_clean"] = df[col].fillna("").apply(
            lambda x: process_multi(x, config["mapping"])
        )

    elif config["type"] == "mixed":
        df[f"{col}_clean"] = df[col].fillna("").apply(
            lambda x: process_row(x, config["mapping"], config["categories"])
        )

    # One-hot encode (only for multi + mixed)
    if config["type"] in ["multi", "mixed"]:
        dummies = df[f"{col}_clean"] \
            .str.get_dummies(sep=";") \
            .add_prefix(f"{col}_")

        all_dummies.append(dummies)

# concat once
df = pd.concat([df] + all_dummies, axis=1)

In [95]:
nlp = spacy.load("en_core_web_sm")

super_categories = [
    "BIPOC",
    "ALICE",
    "Youth (under 18)",
    "Older Adults (+65)",
    "Recovery Community",
    "Town of Residence",
    "Parents",
    "Women",
    "LGBTQ+",
    "Other"
]

# Helper: detect if text contains a town/city
def detect_town(text):
    doc = nlp(text)
    for ent in doc.ents:
        if ent.label_ in ["GPE", "LOC"]:  # geo-political or location
            return True
    return False

# process community
def process_community_cell(text, mapping, super_cats, threshold=0.3):
    if pd.isna(text) or str(text).strip().upper() in ["NA", "N/A", "NONE"]:
        return {"fine": None, "super": None}

    text_clean = str(text).strip()

    # 1️⃣ Exact mapping
    if text_clean in mapping:
        return {
            "fine": mapping[text_clean],
            "super": mapping[text_clean]
        }

    # 2️⃣ Town detection (HARD STOP)
    if detect_town(text_clean):
        return {
            "fine": "Town of Residence",
            "super": "Town of Residence"
        }

    # 3️⃣ Fallback → classifier (SINGLE LABEL ONLY)
    res = classifier(text_clean, candidate_labels=super_cats, multi_label=False)
    label = res['labels'][0] if res['scores'][0] >= threshold else "Other"

    return {
        "fine": "Other",
        "super": label
    }

# Apply to column
results = df['community'].apply(lambda x: process_community_cell(x, mapping_community, super_categories))

# Expand results
df['community_fine'] = results.apply(lambda x: x['fine'])
df['community_super'] = results.apply(lambda x: x['super'])

# One-hot encode super-category
dummies = df['community_super'].str.get_dummies(sep=';').add_prefix("community_")
df = pd.concat([df, dummies], axis=1)

In [96]:
# Adding leading zeroes an making new column
df['zip_code_full'] = df['zip_code'].apply(lambda x: str(int(x)).zfill(5) if pd.notnull(x) else None)

# York County ZIP codes
zip_county_map = {
    "York": [
        "04005", "04072", "04073", "04043", "03909", "04090", "04064", "04093",
        "03904", "04046", "03901", "04002", "03908", "03903", "04027", "03906",
        "04042", "04083", "04049", "04048", "04061", "04076", "04001", "03902",
        "04087", "04030", "04020", "04047", "04095", "03907", "03905", "03805",
        "04063", "03804", "03910", "03911", "04004", "04006", "04007", "04014",
        "04028", "04054", "04056", "04094"
    ],
    "Cumberland": [
        "04103", "04106", "04074", "04011", "04092", "04101", "04062", "04038",
        "04102", "04105", "04107", "04096", "04032", "04084", "04039", "04021",
        "04260", "04009", "04071", "04097", "04055", "04079", "04015", "04040",
        "04085", "04029", "04017", "04069", "04019", "04066", "04091", "04024",
        "04108", "04110", "04050", "04075", "04003", "04013", "04033", "04034",
        "04057", "04070", "04077", "04078", "04082", "04098", "04104", "04109",
        "04112", "04116", "04122", "04123"
    ]
}

#  Function to map ZIP to county
def map_zip_to_county(zip_code):
    if pd.isnull(zip_code):
        return None
    for county, zips in zip_county_map.items():
        if zip_code in zips:
            return county
    return 'Outside Southern Maine'  # or 'Other' if not in these lists

# Apply mapping
df['county'] = df['zip_code_full'].apply(map_zip_to_county)

In [98]:
for col in list(df.columns):
  print(col)

uid
survey_type
start_time
biggest_challenge
live_work_sm
alice
feel_heard
community
age_group
race_ethnicity
relation_uwsm
want_involvement
how_involvement
hardest_expenses
zip_code
employed
employment_type
roadblock_expenses
helpful_supports
biggest_challenge_clean
live_work_sm_clean
alice_clean
feel_heard_clean
age_group_clean
race_ethnicity_clean
relation_uwsm_clean
want_involvement_clean
how_involvement_clean
hardest_expenses_clean
employed_clean
employment_type_clean
roadblock_expenses_clean
helpful_supports_clean
biggest_challenge_Affordable Child Care Access
biggest_challenge_Affordable Elder Care Access
biggest_challenge_Affordable Housing Availability
biggest_challenge_Behavioral Health
biggest_challenge_Climate Impacts
biggest_challenge_Cost of Basics
biggest_challenge_Domestic Violence
biggest_challenge_Education Access
biggest_challenge_Food Access
biggest_challenge_Gas Costs
biggest_challenge_Healthcare Cost
biggest_challenge_Housing Cost
biggest_challenge_Immigration Enf

In [99]:
df.to_csv("processed_data_5.csv", index=False)